# Exercise 02: Performance Evaluation

> **Chapter:** Ch02 - Performance Evaluation
> **Estimated time:** ~55 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

A retrieval metric reduces a ranking to a single number so we can compare systems. Two problems
follow. Different metrics can rank the same two systems in opposite orders, and a number measured on
a test set need not hold once the system runs on real data. You work through both here. First you
explain why average precision and nDCG disagree about the two library systems. Then you build a
reusable `Evaluator`, reproduce the book's numbers with it, and use it to resolve that disagreement.
Last, you examine two cases where a correctly computed number still misleads: a precision that
deployment does not reproduce, and a ranking the benchmark scores as worse when it is better.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] ✏ **T1: Same systems, opposite verdicts** - explain why AP and nDCG disagree
- [ ] 💻 **C1: Build the Evaluator** - implement P@k, AP, MAP, and nDCG (in `tasks.py`)
- [ ] 💻 **C2: The precision you cannot promise** - reason about class balance (in this notebook)
- [ ] ✏ **T2: The ranker that looks worse** - diagnose a benchmark that penalizes a better system

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"02 - Performance Evaluation"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: they ask you to explain
a metric disagreement, build the metrics yourself, and reason about when a correct score lies.

---

## When two metrics disagree

> **How this works:** Write your answer in the markdown cell below each question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you can
> compare and reflect.

Throughout this exercise we reuse the book's running example: a 50-book library, an information need
(*"Which books give a beginning master's student solid computer science foundations?"*) with 15
relevant books graded 3 (core), 2 (specialization), or 1 (peripheral), and two retrieval systems.
**System A** returns 25 books and favours coverage; **System B** returns 8 and favours a clean
result list.

### ✏ Task T1 - Same systems, opposite verdicts

Here are three metrics computed on the CS-foundations need for the two systems, straight from the
book:

| Metric | System A | System B | Winner |
|---|---:|---:|:---:|
| Average Precision (binary, full ranking) | 0.473 | 0.359 | **A** |
| nDCG@10 (graded) | 0.446 | 0.759 | **B** |

These are the same two systems, judged against the same ground truth, yet the metrics disagree about
which one is better. Each metric measures a different property of the ranking, so a system can win on
one and lose on the other.

1. **Explain the disagreement structurally.** Point to the specific part of each formula that drives
   its verdict. Why does AP favour System A (which finds 12 of 15 relevant books but scatters them),
   while nDCG@10 favours System B (which finds only 6 but places its grade-3 books at the very top)?
   Name the role of AP's divisor and of nDCG's cutoff and discount.
2. **Pick a metric for a use case.** A patent lawyer running an exhaustive prior-art search and a
   user skimming the first page of web results should not trust the same metric here. Say which
   metric fits which user, and why choosing the wrong one would mislead that user.
3. **Predict a cutoff change.** You will build an evaluator below. Predict what happens to the
   nDCG *winner* if you recompute at **nDCG@25** instead of nDCG@10, and justify your prediction from
   how the log discount and System B's short result list interact. You will check it against your own
   numbers later in the notebook.

> *Your answer here...*

---

## Setting up

From here you work with the same 50-book library the demos use, together with its graded relevance
judgments and the two systems' ranked runs. Run this cell first. It makes the `shared/` package
importable regardless of where the notebook lives, then loads the collection.

In [ ]:
# Standard setup - run this first
import sys, pathlib, math

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_p = pathlib.Path().resolve()
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

%load_ext autoreload
%autoreload 2

from shared.library_collection import NEEDS, RUNS, run, need_ids, relevant_ids, record
from shared.display import print_table, display_md

# need_id -> {doc_id: grade}. Grade 0 (or absent) means not relevant.
need_grades = {nid: NEEDS[nid]["grades"] for nid in need_ids()}

display_md(
    f"**Needs:** {len(need_grades)}  |  **Primary need:** cs-foundations "
    f"({len(need_grades['cs-foundations'])} relevant books)\n\n"
    f"**System A** returned {len(run('cs-foundations', 'A'))} books; "
    f"**System B** returned {len(run('cs-foundations', 'B'))}."
)

## Building an evaluator

> **How this works:** Implement the `Evaluator` class in `tasks.py`. You can write the code yourself
> or direct an AI to implement it from the spec. After saving `tasks.py`, just re-run the verify
> cell - autoreload picks up your changes automatically, no kernel restart needed.

### 💻 Task C1 - Build the Evaluator

Wrap the chapter's ranked-retrieval metrics into one reusable class so a single object can score any
run against any need. Implement `Evaluator` in `tasks.py` to the following interface. Use the exact
formulas from the chapter (binary AP with divisor $|\text{Rel}|$; graded DCG with the $\log_2(i+1)$
discount; nDCG normalized by the ideal ranking).

- `Evaluator(need_grades: dict[str, dict[str, int]])` - `need_grades` maps a need id to its
  relevance judgments `{doc_id: grade}`. A document absent from a need's dictionary has grade 0
  (not relevant) for that need.
- `precision_at_k(run: list[str], need: str, k: int) -> float` - fraction of the top `k` results
  that are relevant (grade > 0). Positions beyond the end of a short run count as non-relevant, so
  the divisor is always `k`.
- `average_precision(run: list[str], need: str) -> float` - mean of the precision values taken at
  each rank where a relevant document appears, divided by the **total** number of relevant documents
  for the need. A run with no relevant documents scores 0.
- `dcg(run: list[str], need: str, k: int) -> float` - graded discounted cumulative gain over the top
  `k`: $\sum_{i=1}^{k} rel_i / \log_2(i+1)$.
- `ndcg(run: list[str], need: str, k: int) -> float` - `dcg` divided by the ideal DCG at `k` (all of
  the need's grades sorted descending, placed at ranks 1..k). Returns 0 when the ideal is 0.
- `map_score(runs: dict[str, list[str]]) -> float` - macro-average of `average_precision` across the
  needs in `runs` (each key a need id, each value that need's run).

An **empty run** must return 0 from every metric, and `ndcg` of the ideal ranking must be exactly 1.

In [ ]:
# --- Setup C1 ---
from tasks import Evaluator

ev = Evaluator(need_grades)
run_a = run("cs-foundations", "A")
run_b = run("cs-foundations", "B")

# A quick look: P@k for both systems on the primary need.
rows = [[k,
         f"{ev.precision_at_k(run_a, 'cs-foundations', k):.3f}",
         f"{ev.precision_at_k(run_b, 'cs-foundations', k):.3f}"]
        for k in (3, 5, 10)]
print_table(rows, headers=["k", "P@k (A)", "P@k (B)"])

In [ ]:
# --- Verify C1 ---
NEED = "cs-foundations"

# P@k against the book's worked example.
assert abs(ev.precision_at_k(run_a, NEED, 3) - 2/3) < 1e-6, "P@3(A) should be 2/3"
assert abs(ev.precision_at_k(run_a, NEED, 5) - 0.4) < 1e-6, "P@5(A) should be 0.4"
assert abs(ev.precision_at_k(run_b, NEED, 3) - 1.0) < 1e-6, "P@3(B) should be 1.0"
# B returns only 8 docs: positions 9-10 are empty and count as non-relevant.
assert abs(ev.precision_at_k(run_b, NEED, 10) - 0.6) < 1e-6, "P@10(B) should be 6/10 = 0.6"

# Average precision reproduces the book (0.473 for A, 0.359 for B).
ap_a = ev.average_precision(run_a, NEED)
ap_b = ev.average_precision(run_b, NEED)
assert abs(ap_a - 0.473) < 0.01, f"AP(A) should be 0.473, got {ap_a:.4f}"
assert abs(ap_b - 0.359) < 0.01, f"AP(B) should be 0.359, got {ap_b:.4f}"

# Graded nDCG@10 reproduces the book (0.446 for A, 0.759 for B).
ndcg_a = ev.ndcg(run_a, NEED, 10)
ndcg_b = ev.ndcg(run_b, NEED, 10)
assert abs(ndcg_a - 0.446) < 0.01, f"nDCG@10(A) should be 0.446, got {ndcg_a:.4f}"
assert abs(ndcg_b - 0.759) < 0.01, f"nDCG@10(B) should be 0.759, got {ndcg_b:.4f}"

# MAP over all four needs (0.473 for A, 0.740 for B).
runs_a = {nid: run(nid, "A") for nid in need_ids()}
runs_b = {nid: run(nid, "B") for nid in need_ids()}
assert abs(ev.map_score(runs_a) - 0.473) < 0.01, f"MAP(A) got {ev.map_score(runs_a):.4f}"
assert abs(ev.map_score(runs_b) - 0.740) < 0.01, f"MAP(B) got {ev.map_score(runs_b):.4f}"

# Non-obvious property: nDCG of the ideal ranking is exactly 1.
ideal = relevant_ids(NEED)  # relevant docs, highest grade first
assert abs(ev.ndcg(ideal, NEED, 10) - 1.0) < 1e-9, "nDCG of the ideal ranking must be 1.0"

# Edge cases: an empty run scores 0 everywhere.
assert ev.precision_at_k([], NEED, 5) == 0.0
assert ev.average_precision([], NEED) == 0.0
assert ev.ndcg([], NEED, 10) == 0.0

print("✓ Evaluator reproduces the book's numbers and passes the edge cases!")

In [ ]:
# --- Explore C1 ---
# Now settle Task T1 with your own numbers. First, the disagreement made concrete:
display_md("**The disagreement (compute it yourself):**")
print_table(
    [["Average Precision", f"{ev.average_precision(run_a, NEED):.3f}",
      f"{ev.average_precision(run_b, NEED):.3f}"],
     ["nDCG@10", f"{ev.ndcg(run_a, NEED, 10):.3f}", f"{ev.ndcg(run_b, NEED, 10):.3f}"]],
    headers=["Metric", "System A", "System B"],
)

# Now check your T1 part-3 prediction: does the nDCG winner change at cutoff 25?
display_md("**nDCG at growing cutoffs: does the winner flip?**")
print_table(
    [[k, f"{ev.ndcg(run_a, NEED, k):.3f}", f"{ev.ndcg(run_b, NEED, k):.3f}",
      "A" if ev.ndcg(run_a, NEED, k) > ev.ndcg(run_b, NEED, k) else "B"]
     for k in (5, 10, 15, 25)],
    headers=["k", "nDCG@k (A)", "nDCG@k (B)", "Winner"],
)

## When a correct number still lies

You now have metrics you trust. The remaining tasks show how such a metric can mislead even when it is
computed correctly: a score can describe the test set faithfully and still not describe the system you
deploy.

> **How this works:** Implement the function in the stub cell. Run the verify cell; the assertions
> either pass or fail with a specific message. Then read the explore cell's output and think about
> what it shows.

### 💻 Task C2 - The precision you cannot promise

A classifier's **recall** (true positive rate) and **specificity** (true negative rate) are
properties of the classifier: they measure how it treats positive and negative items and do not
depend on how many of each exist. **Precision** is different. It mixes true and false positives, and
false positives are drawn from the negative pool, so precision depends on the **class balance** of the
data it runs on.

Implement `precision_from_rates`, which computes the precision a classifier would show at a given
**prevalence** (the fraction of items that are truly positive), from its recall and specificity alone.
Consider a large population of items: a fraction `prevalence` are positive and the rest negative. Of
the positives, a fraction `recall` become true positives; of the negatives, a fraction
`(1 - specificity)` become false positives. Precision is true positives over all predicted positives.

In [ ]:
def precision_from_rates(recall: float, specificity: float, prevalence: float) -> float:
    """Precision implied by a classifier's recall and specificity at a given prevalence."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Verify C2 ---
# The book's spam filter: recall 0.90, specificity 0.98.
# On a balanced test set it looks excellent; in a 5%-spam mailbox it is much worse.
p_balanced = precision_from_rates(0.90, 0.98, 0.50)
p_deployed = precision_from_rates(0.90, 0.98, 0.05)
assert abs(p_balanced - 0.978) < 0.001, f"balanced precision got {p_balanced:.4f}"
assert abs(p_deployed - 0.703) < 0.001, f"deployed precision got {p_deployed:.4f}"

# No positives in the population -> no true positives -> precision defined as 0.
assert precision_from_rates(0.90, 0.98, 0.0) == 0.0, "prevalence 0 should give precision 0"

# Non-obvious property: with specificity < 1, precision strictly falls as positives get rarer.
assert p_balanced > p_deployed, "precision should drop as prevalence drops"

print("✓ precision_from_rates correct!")

In [ ]:
# --- Explore C2 ---
# Sweep prevalence for the fixed classifier (recall 0.90, specificity 0.98) and watch
# precision collapse even though the classifier never changed.
display_md("**Same classifier, falling prevalence:**")
print_table(
    [[f"{p:.0%}", f"{precision_from_rates(0.90, 0.98, p):.3f}"]
     for p in (0.50, 0.20, 0.10, 0.05, 0.02, 0.01)],
    headers=["Prevalence (spam)", "Precision"],
)
# Recall stays 0.90 and specificity stays 0.98 at every row above. Only the precision the user
# actually experiences moves. A test set must therefore match deployment prevalence, or the
# reported precision will not hold in production.

### ✏ Task T2 - The ranker that looks worse

You build a new neural ranker and evaluate it on a widely used, several-years-old TREC collection by
computing MAP against the collection's stored relevance judgments. Your ranker scores a **lower** MAP
than the old keyword baseline everyone cites. Yet when you read its top-10 results by hand for a dozen
needs, they look better: on-topic, well-ordered, and surfacing good documents the baseline misses
entirely.

1. Explain how a genuinely better ranker can score **lower** MAP on such a collection. What property of
   how these benchmarks were judged produces the effect, and why does it hit your *new* system rather
   than the baseline that helped build the collection?
2. Explain why **MAP** and **recall** are hit harder by this than **P@10** would be.
3. Propose a concrete way to get a fair comparison, and note one limit of your fix.

> *Your answer here...*

---

## Summary

In [ ]:
# Run this cell last to confirm everything passed.
print("Ch02 exercise complete!")
print("  Quiz:          done (check your score in the app)")
print("  Text tasks:    T1, T2 - compare with the solution notebook")
print("  Helper task:   C1 (Evaluator) ✓")
print("  Inline coding: C2 (precision_from_rates) ✓")